In [ ]:
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import os
import hashlib
from datetime import datetime

import time

# ==========================================
# CUSTOMIZATION PARAMETERS (MODIFY THESE)
# ==========================================

# Domain parameters
xc = 0
yc = 0
W = 1
H = 1
nx = 2000
ny = 2000
d = 0.02#0.005  # initial pulse width

# Wave parameters
c = 1.0  # wave speed
steps = 500
cfl_factor = 0.5  # CFL stability condition factor

# Damping parameters
absorb_width = 0.1
absorb_strength = 22.0

# Source parameters
use_ext_source = False  # set to True to enable external source
use_const_source = False  # set to True to use constant source from initial condition

# Particle parameters
mass = 0.01
charge = 0.01

# Animation parameters
frame_step = 20  # use every 10th frame
fps = 50
figsize = (5, 5)
color_factor = 1.125 # Used to cap the colorbar's maximum and minimum color based on the average values of the entire simulation.
animation_interval = 10  # ms between frames

# Output parameters
output_subfolder = 'Wave Simulations'
bitrate = 3000

# ==========================================
# INITIAL CONDITION
# Modify this to change the initial wave pattern.
# ==========================================

def initial_field(X, Y, x_center, y_center, pulse_width):
    #radius = (X - x_center)**2 + (Y - y_center)**2
    return np.zeros_like(X) #np.exp(-radius/(pulse_width)**2)

# ==========================================
# LAPLACIAN OPERATOR
# 9-point stencil to minimize anisotropy of the difference operations.
# ==========================================

def get_laplace(Z, dx):
    Zp = np.pad(Z, 1, mode='edge')
    
    return (
        4 * (
            Zp[2:,1:-1] +
            Zp[:-2,1:-1] +
            Zp[1:-1,2:] +
            Zp[1:-1,:-2]
        )
        +
        (
            Zp[2:,2:] +
            Zp[2:,:-2] +
            Zp[:-2,2:] +
            Zp[:-2,:-2]
        )
        -
        20 * Z
    ) / (6 * dx**2)

# ==========================================
# MATH HELPERS
# ==========================================

def bilinear_interpolate(x, y, F, px, py): #Get the interpolated value of a field in a point that is between four points of a coordinate grid.
    """
    x, y : 1D coordinate arrays
    F    : 2D field array with shape (len(y), len(x))
    px, py : interpolation point
    """

    # Find indices of surrounding cell
    ix = np.searchsorted(x, px) - 1
    iy = np.searchsorted(y, py) - 1

    # Clamp to valid range
    ix = np.clip(ix, 0, len(x)-2)
    iy = np.clip(iy, 0, len(y)-2)

    # Corner coordinates
    x1, x2 = x[ix], x[ix+1]
    y1, y2 = y[iy], y[iy+1]

    # Corner values
    Q11 = F[iy, ix]
    Q21 = F[iy, ix+1]
    Q12 = F[iy+1, ix]
    Q22 = F[iy+1, ix+1]

    # Relative coordinates inside cell
    tx = (px - x1) / (x2 - x1)
    ty = (py - y1) / (y2 - y1)

    # Bilinear interpolation
    value = (
        (1-tx)*(1-ty)*Q11 +
        tx*(1-ty)*Q21 +
        (1-tx)*ty*Q12 +
        tx*ty*Q22
    )

    return value

def bilinear_gradient(x, y, F, px, py):
    dx = x[1] - x[0]
    dy = y[1] - y[0]

    dFdx = np.gradient(F, dx, axis=1)
    dFdy = np.gradient(F, dy, axis=0)

    gx = bilinear_interpolate(x, y, dFdx, px, py)
    gy = bilinear_interpolate(x, y, dFdy, px, py)

    return gx, gy

# ==========================================
# PARTICLES
# ==========================================

def set_particle_distribution(simulation):
    distribution = simulation['distribution']
    if distribution['type'] == "none":
        simulation["x_pos_init"] = [simulation['sim_center'][0]]
        simulation["y_pos_init"] = [simulation['sim_center'][1]]
        return simulation
        
    xc, yc = distribution['center']
    num_particles = simulation["num_particles"]
    if num_particles == 1:
        simulation["x_pos_init"] = [xc]
        simulation["y_pos_init"] = [yc]
        return simulation

    x_particles = []
    y_particles = []
    
    if distribution['type'] == "line":
        length = distribution['length']
        angle = distribution['angle'] * np.pi/180
        xr = np.cos(angle) * length
        yr = np.sin(angle) * length
        for i in range(num_particles):
            new_x = xc + xr * (i - (num_particles - 1)/2)/(num_particles-1)
            new_y = yc + yr * (i - (num_particles - 1)/2)/(num_particles-1)
            x_particles.append(new_x)
            y_particles.append(new_y)
        simulation["x_pos_init"] = x_particles
        simulation["y_pos_init"] = y_particles
        return simulation
        
    if distribution['type'] == "ring":
        radius = distribution['length']/2
        angle = distribution['angle'] * np.pi/180
        proportion = distribution['proportion']
        rad_x = radius * proportion
        rad_y = radius / proportion
        for i in range(num_particles):
            new_x = xc + rad_x * np.cos(2*np.pi * i/num_particles + angle)
            new_y = yc + rad_y * np.sin(2*np.pi * i/num_particles + angle)
            x_particles.append(new_x)
            y_particles.append(new_y)
        simulation["x_pos_init"] = x_particles
        simulation["y_pos_init"] = y_particles
        return simulation

# ==========================================
# PARTICLE DYNAMICS
# ==========================================

def particle_field(index, simulation):
    X = simulation['X']
    Y = simulation['Y']
    time = simulation['time']
    x_pos = simulation['x_pos_new'][index]
    y_pos = simulation['y_pos_new'][index]
    z_pos = simulation['z_pos_new'][index]
    pulse_width = simulation['pulse_width_new'][index]
    
    radius = (X - x_pos)**2 + (Y - y_pos)**2 + z_pos ** 2
    return np.exp(-radius/(pulse_width)**2)

def linear_motion(index, simulation):
    time = simulation["time"]
    params = simulation["motion_params"][index]
    x_inc = 0
    y_inc = 0
    if time > params["lin_delay"] and time < params["lin_end"]:
        x_inc = params["velocity"][0] * simulation["dt"]
        y_inc = params["velocity"][1] * simulation["dt"]

    return x_inc, y_inc

def quadratic_motion(index, simulation): 
    time = simulation["time"]
    params = simulation["motion_params"][index]
    x_inc = 0
    y_inc = 0
    if time > params["quad_delay"] and time < params["quad_end"]:
        accel_x = params["acceleration"][0]
        accel_y = params["acceleration"][1]
        x_inc = accel_x / 2 * simulation["dt"] * (time - params["quad_delay"])
        y_inc = accel_y / 2 * simulation["dt"] * (time - params["quad_delay"])
    return x_inc, y_inc

def oscillatory_motion(index, simulation):
    time = simulation["time"]
    params = simulation["motion_params"][index]
    x_inc = 0
    y_inc = 0
    z_inc = 0
    if time > params["osc_delay"] and time < params["osc_end"]:
        if not ("angular_frequency") in simulation["motion_params"][index]:
            freq_x = params["frequency"][0]
            freq_y = params["frequency"][1]
            freq_z = params["frequency"][2]
            simulation["motion_params"][index]["angular_frequency"] = [2*np.pi*freq_x, 2*np.pi*freq_y, 2*np.pi*freq_z]
            params = simulation["motion_params"][index]
        ang_freq_x = params["angular_frequency"][0]
        ang_freq_y = params["angular_frequency"][1]
        ang_freq_z = params["angular_frequency"][2]

        if not ("oscillation_velocity") in simulation["motion_params"][index]:
            rad_x = params["amplitude"][0] #Peak amplitude in x-axis
            rad_y = params["amplitude"][1] #Peak amplitude in y-axis
            rad_z = params["amplitude"][2]#Peak amplitude in z-axis
            simulation["motion_params"][index]["oscillation_velocity"] = [ang_freq_x*rad_x, ang_freq_y*rad_y, ang_freq_z*rad_z]
            params = simulation["motion_params"][index]
        osc_vel_x = params["oscillation_velocity"][0]
        osc_vel_y = params["oscillation_velocity"][1]
        osc_vel_z = params["oscillation_velocity"][2]
        
        phase_x = params["phase"][0]
        phase_y = params["phase"][1]
        phase_z = params["phase"][2]
        
        x_inc = osc_vel_x * np.cos(ang_freq_x * time - phase_x) * simulation["dt"]
        y_inc = osc_vel_y * np.cos(ang_freq_y * time - phase_y) * simulation["dt"]
        z_inc = osc_vel_z * np.cos(ang_freq_z * time - phase_z) * simulation["dt"]

    return x_inc, y_inc, z_inc

def particle_motion_by_forcefield(index, simulation): #The total field accelerates the particles.
    current_x = simulation['x_pos'][index]
    current_y = simulation['y_pos'][index]
    dz = 0
    field = simulation["current_field"].copy()
    if simulation["prevent_self_interaction"]:
        field -= particle_field(index, simulation)
    x = simulation['X'][0,:]  # Get x coordinates from grid
    y = simulation['Y'][:,0]  # Get y coordinates from grid
    field_strength = bilinear_interpolate(x, y, field, current_x, current_y)
    dz = field_strength * simulation["particle_charges"][index] / simulation["particle_masses"][index] * simulation["dt2"]
    return dz

def update_particle_motion(index, simulation):
    simulation['x_pos_new'][index] = simulation['x_pos'][index]
    simulation['y_pos_new'][index] = simulation['y_pos'][index]
    simulation['z_pos_new'][index] = simulation['z_pos'][index]
    x_inc = 0
    y_inc = 0
    z_inc = 0

    if "stationary" in simulation["motion_params"][index]["motion_type"]:
        x_inc, y_inc = 0, 0
        stationary = True
    else:
        if "linear" in simulation["motion_params"][index]["motion_type"]:
            x_inc, y_inc = linear_motion(index, simulation)  # Fixed assignment
        if "quadratic" in simulation["motion_params"][index]["motion_type"]:
            x_inc2, y_inc2 = quadratic_motion(index, simulation)  # Fixed
            x_inc += x_inc2
            y_inc += y_inc2
        if "oscillatory" in simulation["motion_params"][index]["motion_type"]:
            x_inc3, y_inc3, z_inc3 = oscillatory_motion(index, simulation)  # Fixed
            x_inc += x_inc3
            y_inc += y_inc3
            z_inc += z_inc3
        if "field" in simulation["motion_params"][index]["motion_type"]:  # Added missing colon
            z_inc = particle_motion_by_forcefield(index, simulation)
        
    if simulation["prevent_superluminal"]:
        sq_ratio = (x_inc**2 + y_inc**2 + z_inc**2) / (simulation["c"] * simulation["dt"])**2
        if sq_ratio >= 1:  # Added missing colon
            print(f"WARNING: particle {index} at time {simulation['time']} would exceed wave speed. Capping speed.")
            slowdown = 1 / np.sqrt(sq_ratio) * simulation["speed_cap"]
            x_inc *= slowdown
            y_inc *= slowdown
            z_inc *= slowdown
        
    simulation['x_pos_new'][index] += x_inc
    simulation['y_pos_new'][index] += y_inc
    simulation['z_pos_new'][index] += z_inc
    simulation['x_pos'][index] = simulation['x_pos_new'][index]
    simulation['y_pos'][index] = simulation['y_pos_new'][index]
    simulation['z_pos'][index] = simulation['z_pos_new'][index]
    return simulation

def set_default_motion_params(num_particles, default_type, dt):
    motion_params = []
    for i in range(num_particles):
        if default_type == "z-wave":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : np.inf,
                    "quad_end" : np.inf,
                    "osc_end" : np.inf,
                    "phase" : [0, 0, 0],#2*np.pi*i/num_particles],
                    "frequency" : [0, 0, 30],
                    "amplitude" : [0, 0, 0.001],
                }
        elif default_type == "x-wave":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : np.inf,
                    "quad_end" : np.inf,
                    "osc_end" : np.inf,
                    "phase" : [2*np.pi*i/num_particles, 0, 0],
                    "frequency" : [2, 0, 0],
                    "amplitude" : [0.02, 0, 0],
                }
        elif default_type == "x-wiggle":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : np.inf,
                    "quad_end" : np.inf,
                    "osc_end" : np.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [1, 0, 0],
                    "amplitude" : [0.1, 0, 0],
                }
        elif default_type == "circle":
            params = {
                    "motion_type" : ["oscillatory"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : np.inf,
                    "quad_end" : np.inf,
                    "osc_end" : np.inf,
                    "phase" : [2*np.pi*i/num_particles  + np.pi*i, 2*np.pi*i/num_particles - np.pi / 2 + np.pi*i, 0],
                    "frequency" : [1, 1, 0],
                    "amplitude" : [0.1, 0.1, 0],
                }
        elif default_type == "linear":
            params = {
                    "motion_type" : ["linear"],
                    "velocity" : [0.1, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : i * 15 * dt,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : np.inf,
                    "quad_end" : np.inf,
                    "osc_end" : np.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [0, 0, 0],
                    "amplitude" : [0, 0, 0],
                }
        elif default_type == "accelerate":
            params = {
                    "motion_type" : ["quadratic"],
                    "velocity" : [0, 0],
                    "acceleration" : [0.1, 0],
                    "lin_delay" : 0,
                    "quad_delay" : i * 1 * dt,
                    "osc_delay" : 0,
                    "lin_end" : np.inf,
                    "quad_end" : np.inf,
                    "osc_end" : np.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [0, 0, 0],
                    "amplitude" : [0, 0, 0],
                }
        elif default_type == "interact":
            params = {
                    "motion_type" : ["field"],
                    "velocity" : [0, 0],
                    "acceleration" : [0, 0],
                    "lin_delay" : 0,
                    "quad_delay" : 0,
                    "osc_delay" : 0,
                    "lin_end" : np.inf,
                    "quad_end" : np.inf,
                    "osc_end" : np.inf,
                    "phase" : [0, 0, 0],
                    "frequency" : [0, 0, 0],
                    "amplitude" : [0, 0, 0],
                }
        elif default_type == "orbit":
            params = {
                "motion_type" : ["linear", "interact"],
                "velocity" : [0.01 * (-1)**i, 0],
                "acceleration" : [0, 0],
                "lin_delay" : 0,
                "quad_delay" : 0,
                "osc_delay" : 0,
                "lin_end" : np.inf,
                "quad_end" : np.inf,
                "osc_end" : np.inf,
                "phase" : [0, 0],
                "frequency" : [0, 0, 0],
                "radius" : [0, 0, 0],
            }
        else:
            params = {
                "motion_type" : ["stationary"],
                "velocity" : [0, 0],
                "acceleration" : [0, 0],
                "lin_delay" : 0,
                "quad_delay" : 0,
                "osc_delay" : 0,
                "lin_end" : np.inf,
                "quad_end" : np.inf,
                "osc_end" : np.inf,
                "phase" : [0, 0],
                "frequency" : [0, 0, 0],
                "radius" : [0, 0, 0],
            }
        motion_params.append(params)
    return motion_params

def run_simulation():
    """Run simulation with:
        - dynamic wave field
        - static/moving particle near-fields
        - radiation from particle motion via delta_particle
    """

    # ==========================================
    # GRID
    # ==========================================

    x = np.linspace(xc - W/2, xc + W/2, nx)
    y = np.linspace(yc - H/2, yc + H/2, ny)

    X, Y = np.meshgrid(x, y)

    dx = x[1] - x[0]

    # CFL stability condition
    dt = cfl_factor * dx / c

    # ==========================================
    # INITIAL WAVE FIELD
    # ==========================================

    Z0 = initial_field(X, Y, xc, yc, d)

    # ONLY the propagating/radiative component evolves
    Z_wave = Z0.copy()
    Z_wave_prev = Z0.copy()

    # ==========================================
    # CONSTANT SOURCE
    # ==========================================

    const_source = (
        -c**2 * get_laplace(Z0, dx)
        if use_const_source else 0
    )

    # ==========================================
    # ABSORBING BOUNDARY
    # ==========================================

    R = np.sqrt((X - xc)**2 + (Y - yc)**2)

    Rmax = min(W, H) / 2
    R0 = Rmax - absorb_width

    s = np.clip((R - R0) / absorb_width, 0, 1)

    damping = np.exp(-absorb_strength * s**2)

    sigma_max = 80.0

    sigma = np.zeros_like(R)

    sigma[s > 0] = sigma_max * s[s > 0]**4

    # ==========================================
    # PARTICLES
    # ==========================================

    conjugate_charges = False

    num_particles = 2

    motion_params = set_default_motion_params(
        num_particles,
        "z-wave",
        dt
    )

    distribution = {
        "type": "ring",
        "center": [xc, yc],
        "length": 0.05,
        "angle": 90,
        "proportion": 1,
    }

    simulation = {

        "X": X,
        "Y": Y,

        "time": 0,

        "dt": dt,
        "dt2": dt**2,

        "dx": dx,

        "c": c,

        "source_balance": True,

        "sim_center": [xc, yc],

        "num_particles": num_particles,

        "prevent_superluminal": True,
        "speed_cap": 0.99,

        "prevent_self_interaction": True,

        "current_field": np.zeros_like(Z0),

        "x_pos_init": np.zeros(num_particles),
        "y_pos_init": np.zeros(num_particles),
        "z_pos_init": np.zeros(num_particles),

        "distribution": distribution,

        "pulse_width_init": np.ones(num_particles) * d,

        "particle_masses": np.ones(num_particles) * mass,
        "particle_charges": np.ones(num_particles) * charge,

        "x_pos": np.zeros(num_particles),
        "y_pos": np.zeros(num_particles),
        "z_pos": np.zeros(num_particles),

        "x_pos_new": np.zeros(num_particles),
        "y_pos_new": np.zeros(num_particles),
        "z_pos_new": np.zeros(num_particles),

        "pulse_width_new": np.ones(num_particles) * d,

        "motion_params": motion_params
    }

    if conjugate_charges:
        for i in range(num_particles):
            simulation["particle_charges"][i] *= (-1)**i

    # ==========================================
    # INITIALIZE PARTICLE POSITIONS
    # ==========================================

    simulation = set_particle_distribution(simulation)

    for i in range(num_particles):

        simulation["x_pos"][i] = simulation["x_pos_init"][i]
        simulation["y_pos"][i] = simulation["y_pos_init"][i]
        simulation["z_pos"][i] = simulation["z_pos_init"][i]

    # ==========================================
    # INITIAL PARTICLE FIELD
    # ==========================================

    particle_previous = np.zeros_like(Z_wave)

    for i in range(num_particles):

        particle_previous += (
            simulation["particle_charges"][i]
            * particle_field(i, simulation)
        )

    # initialize displayed field consistently
    initial_particle_field = particle_previous.copy()
    
    # optional:
    # include particles in initial displayed state
    frames_initial = Z_wave + initial_particle_field
    
    # ==========================================
    # OUTPUT STORAGE
    # ==========================================

    frames_file = "wave_frames.dat"

    frames = np.memmap(
        frames_file,
        dtype=np.float32,
        mode='w+',
        shape=(steps, ny, nx)
    )

    # ==========================================
    # RADIATION STRENGTH
    # ==========================================

    radiation_strength = 1.0

    # ==========================================
    # MAIN LOOP
    # ==========================================
    for step in range(steps):
        simulation["time"] = step * dt

        # --------------------------------------
        # UPDATE PARTICLE POSITIONS
        # --------------------------------------

        for i in range(num_particles):

            simulation["current_field"] = Z_wave

            simulation = update_particle_motion(
                i,
                simulation
            )

        # --------------------------------------
        # BUILD CURRENT PARTICLE FIELD
        # --------------------------------------

        particle_total = np.zeros_like(Z_wave)

        for i in range(num_particles):

            particle_total += (
                simulation["particle_charges"][i]
                * particle_field(i, simulation)
            )

        # --------------------------------------
        # PARTICLE FIELD CHANGE
        # --------------------------------------

        if step == 0:
            delta_particle = np.zeros_like(Z_wave)
        else:
            delta_particle = (
                particle_total
                - particle_previous
            )

        # --------------------------------------
        # WAVE VELOCITY
        # --------------------------------------

        velocity = Z_wave - Z_wave_prev

        # --------------------------------------
        # OPTIONAL EXTERNAL SOURCE
        # --------------------------------------

        source_term = 0

        if use_ext_source:
            source_term = dt**2 * ext_source(X, Y)

        # --------------------------------------
        # EVOLVE DYNAMIC WAVE FIELD
        # --------------------------------------

        Z_wave_next = (
            2 * Z_wave
            - Z_wave_prev
            + (c * dt)**2 * get_laplace(Z_wave, dx)
            + source_term
            + const_source * dt**2
            + radiation_strength * delta_particle
            - sigma * velocity * dt
        )

        # absorbing boundary

       # Z_wave_next *= damping

        # --------------------------------------
        # TOTAL OBSERVED FIELD
        # --------------------------------------

        Z_total = (
            Z_wave_next
            + particle_total
        )
        Z_total *= damping
        # --------------------------------------
        # STORE FRAME
        # --------------------------------------

        frames[step] = Z_wave_next.astype(np.float32)

        # --------------------------------------
        # ADVANCE STATE
        # --------------------------------------

        Z_wave_prev = Z_wave
        Z_wave = Z_wave_next

        particle_previous = particle_total.copy()

        # --------------------------------------
        # DIAGNOSTICS
        # --------------------------------------

        if step % 50 == 0:

            total_field_energy = np.sum(
                Z_wave**2
                + (velocity / c)**2
            ) * dx**2
         #   print(f"Step {step}", f"Field energy: {total_field_energy:.5e}")

    # ==========================================
    # CLEANUP
    # ==========================================


    frames.flush()

    frames._mmap.close()

    del frames
    
    return frames_file, x, y, dt

def generate_filename(dt):
    """Generate unique filename based on current parameters"""
    
    # Create a string with key parameters
    param_str = (
        f"W{W}_H{H}_nx{nx}_ny{ny}_d{d}_"
        f"c{c}_steps{steps}_cfl{cfl_factor}_"
        f"abs{absorb_strength}_{absorb_width}_"
        f"ext{use_ext_source}_const{use_const_source}_"
        f"dt{dt:.6f}"
    )
    
    # Create hash for uniqueness
    param_hash = hashlib.md5(param_str.encode()).hexdigest()[:8]
    
    # Add timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    filename = f"wave_{param_hash}_{timestamp}.mp4"
    
    return filename

def create_animation(frames_file, x, y, dt):
    """Create and save animation"""

    frames = np.memmap(
        frames_file,
        dtype=np.float32,
        mode='r',
        shape=(steps, ny, nx)
    )
        
    # Select frames
    display_frames = frames[::frame_step]
    display_dt = dt * frame_step
    
    # Calculate color limits
    vmin = np.inf
    vmax = -np.inf
    
    for i in range(0, steps, frame_step):
        vmin = min(vmin, frames[i].min())
        vmax = max(vmax, frames[i].max())
    
    vmin *= color_factor
    vmax *= color_factor
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    im = ax.imshow(
        display_frames[0],
        extent=[x[0], x[-1], y[0], y[-1]],
        origin='lower',
        vmin=vmin,
        vmax=vmax,
        animated=True,
        cmap='RdBu_r'
    )
    
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('z')
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    
    fig.tight_layout(pad=0.2)
    
    # Update function
    def update(frame):
        current_Z = frames[frame * frame_step]
        im.set_array(current_Z)
        ax.set_title(f't = {display_dt*frame:.5f}s')
        return [im]
        
    # Create animation
    print("Creating animation...")
    ani = FuncAnimation(
        fig,
        update,
        frames=len(display_frames),
        interval=animation_interval,
        blit=True
    )
    
    # Generate filename and save
    filename = generate_filename(dt)
    
    # Create subfolder if needed
    if not os.path.exists(output_subfolder):
        os.makedirs(output_subfolder)
        print(f"Created directory: {output_subfolder}")
    
    filepath = os.path.join(output_subfolder, filename)
    
    # Save
    print(f"Saving animation to: {filepath}")
    writer = FFMpegWriter(fps=fps, bitrate=bitrate)
    ani.save(filepath, writer=writer)
    
    print(f"Animation saved successfully!")
    
    # Show final frame
    plt.show()

    frames._mmap.close()
    del frames
    return ani

# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    # Set parameters before running    
    # Run simulation
    t_start = time.time()
    frames, x, y, dt = run_simulation()
    t_sim = time.time()
    sim_total_time = t_sim - t_start
    sim_frame_avg = sim_total_time/steps
    sim_point_avg = sim_frame_avg / (nx * ny)
    print(f"Time to calculate simulation: {(sim_total_time):.4f} seconds;\nAverage per frame: {(sim_frame_avg * 1e3):.4f} milliseconds; \nAverage per point: {(sim_point_avg*1e6):.4f} microseconds.")
    
    # Create and save animation
    t_ani0 = time.time()
    #ani = create_animation(frames, x, y, dt)
    t_ani = time.time()
   # print(f"Time to build animation: {(t_ani - t_ani0):.4f} seconds")
